# Day 4 — Guardrails, Policy Validation & Human-in-the-Loop

## Objective

Make the Airport Operations AI Copilot safe enough to handle
operational actions by introducing:

- Input validation
- Output validation
- Risk classification
- Policy enforcement
- Human approval
- Audit logging
- Distilled training data

## Safety Architecture

User Request
     ↓
Input Guardrail
     ↓
Policy Guardrail
     ↓
Risk Classification
     ↓
Approval Required?
   ├── No → Execute
   └── Yes
          ↓
     Human Approval
       ├── Reject → Stop
       └── Approve → Execute
                         ↓
                  Output Guardrail
                         ↓
                    Audit Log

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.guardrails import (
    validate_action_input,
    validate_tool_output,
    validate_policy_action,
    classify_action,
)

from src.approval import evaluate_approval
from src.action_controller import execute_guarded_action

## Risk Classification

Project assumptions:

| Action | Risk | Approval |
|---|---|---|
| Read metrics | Low | No |
| Search policy | Low | No |
| Calculate incentive | Medium | No/Conditional |
| Increase surge < 1.3x | Medium | Yes |
| Increase surge >= 1.3x | High | Yes |
| Incentive above $3,000 | High | Yes |

These thresholds are synthetic assumptions for this project.

In [3]:
classify_action(
    action="trigger_surge_override",
    new_multiplier=1.4
)

{'status': 'success',
 'action': 'trigger_surge_override',
 'risk_level': 'high',
 'approval_required': True}

In [4]:
validate_action_input(
    action="trigger_surge_override",
    arguments={
        "airport_code": "SFO",
        "new_multiplier": 1.4,
        "reason": "High airport demand"
    }
)

{'valid': True,
 'arguments': {'airport_code': 'SFO',
  'new_multiplier': 1.4,
  'reason': 'High airport demand'}}

In [5]:
validate_policy_action(
    action="trigger_surge_override",
    arguments={
        "airport_code": "SFO",
        "new_multiplier": 2.0,
        "reason": "High demand"
    }
)

{'valid': False,
 'policy_violation': True,
 'message': 'Policy violation: SFO maximum surge is 1.5x, but 2.0x was requested.'}

### Policy Enforcement

The synthetic SFO policy allows a maximum surge multiplier of 1.5x.

A request for 2.0x is therefore blocked even though 2.0x
is within the general system safety limit of 2.0x.

This demonstrates that an AI recommendation cannot override
a higher-priority business policy.

In [6]:
risk_info = classify_action(
    action="trigger_surge_override",
    new_multiplier=1.4
)

evaluate_approval(
    risk_info=risk_info,
    human_approved=None
)

{'status': 'pending',
 'approved': False,
 'risk_level': 'high',
 'message': 'Human approval is required before execution.'}

In [7]:
evaluate_approval(
    risk_info=risk_info,
    human_approved=True
)

{'status': 'approved',
 'approved': True,
 'risk_level': 'high',
 'message': 'Human approval received. Action may proceed.'}

In [8]:
evaluate_approval(
    risk_info=risk_info,
    human_approved=False
)

{'status': 'rejected',
 'approved': False,
 'risk_level': 'high',
 'message': 'Human approval rejected. Action must not execute.'}

In [9]:
result = execute_guarded_action(
    action="trigger_surge_override",
    arguments={
        "airport_code": "SFO",
        "new_multiplier": 1.4,
        "reason": "High airport demand"
    },
    user_request="Increase SFO surge because airport demand is high"
)

result

{'status': 'blocked',
 'stage': 'human_approval',
 'action': 'trigger_surge_override',
 'risk_level': 'high',
 'approval': {'status': 'pending',
  'approved': False,
  'risk_level': 'high',
  'message': 'Human approval is required before execution.'},
 'message': 'Human approval is required before execution.'}

## Audit Trail

Every guarded action should capture:

- User request
- Agents invoked
- Tools called
- Retrieved policies
- Recommendation
- Risk level
- Approval decision
- Final action
- Execution result

Audit records are stored in:

`output/audit_log.jsonl`

## Distilled Training Data

Successful AI interactions can be stored as structured JSONL
records for future evaluation or model improvement.

Fine-tuning is not required for this project.

Output:

`output/distilled_training_data.jsonl`

In [11]:
from pathlib import Path

distillation_file = (
    PROJECT_ROOT /
    "output" /
    "distilled_training_data.jsonl"
)

print(distillation_file)
print(distillation_file.exists())

/home/nineleaps/Documents/airport-ai-copilot/output/distilled_training_data.jsonl
True


# Day 4 Deliverables

✓ Input validation

✓ Output validation

✓ Risk classification

✓ Policy validation

✓ Human-in-the-loop approval

✓ Invalid actions blocked

✓ Policy-violating actions blocked

✓ High-risk actions require approval

✓ Audit logging

✓ Distilled training data

## Success Criteria

The system can:

1. Block invalid inputs.
2. Block policy-violating actions.
3. Require human approval for high-risk actions.
4. Record the decision and execution trail.